# Load records with a batch
Load JSON-LD into Blue Core the way bulk loads do: upload a file (`/batches/upload/`) or point at one by URI (`/batches/`). The API hands it to an Airflow DAG (`resource_loader`, or `archived_file_loader` for a `.zip` or `.tar.gz`), and this notebook follows the run to the end and finds the records it made.

**Local stack only.** It creates records and deletes them again at the end, so it stops on dev and stage, which are shared. Signing in uses your stack's API login (`API_KEYCLOAK_USER`, from the stack checkout), with the API's `create` and `update` roles.

In [ ]:
import json, uuid
from bluecore import show, target
bc = target()
bc.require_writable()               # stops here on dev and stage
AUTH = bc.auth_headers()            # a Keycloak token for your stack's API login
LD = {**AUTH, "Content-Type": "application/ld+json"}   # a raw JSON-LD body
RUN = uuid.uuid4().hex[:8]          # marks this run's records: https://example.org/notebook/<RUN>/...
bc

## A file to load
Two example works, made from the API's request example. Use your own by setting `FILE` to a `.jsonld` (or `.zip`) in this notebook's folder.

In [ ]:
import pathlib

FILE = ""   # e.g. "my-records.jsonld"
LABEL = f"Notebook batch {RUN}"   # in every example record, to find them afterwards

if FILE:
    name, content = pathlib.Path(FILE).name, pathlib.Path(FILE).read_bytes()
else:
    works = [{**bc.example("/works/"), "@id": f"https://example.org/notebook/{RUN}/work-{i}", "rdfs:label": f"{LABEL} #{i}"} for i in (1, 2)]
    name, content = f"notebook-{RUN}.jsonld", json.dumps({"@context": works[0]["@context"], "@graph": [{k: v for k, v in w.items() if k != "@context"} for w in works]}).encode()
print(name, len(content), "bytes")

## Upload it
The answer names the DAG run (`workflow_id`).

In [ ]:
started = bc.get("/api/change_documents/works/feed").json()["totalItems"]
r = bc.post("/api/batches/upload/", files={"file": (name, content)}, headers=AUTH)
show(r)
batch = r.json()
DAG = "archived_file_loader" if name.endswith((".zip", ".tar.gz", ".tgz", ".gz")) else "resource_loader"

## Follow the DAG run

In [ ]:
import pandas as pd

state, tasks = bc.wait_for_run(DAG, batch["workflow_id"], timeout=600)
print(DAG, batch["workflow_id"], "→", state)
pd.DataFrame(tasks)

In [ ]:
if state != "success":
    s, api = bc.airflow()
    for t in tasks:
        if t["state"] == "failed":
            log = s.get(f"{api}/dags/{DAG}/dagRuns/{batch['workflow_id']}/taskInstances/{t['task']}/logs/1", headers={"Accept": "text/plain"})
            print(f"--- {t['task']} ---")
            print(log.text[-3000:])
else:
    print("Loaded.")

## What it made
New entries in the works change feed since the upload.

In [ ]:
feed = bc.get("/api/change_documents/works/feed").json()
new = feed["totalItems"] - started
last = int(feed["last"]["id"].rsplit("/", 1)[1])
items = bc.get(f"/api/change_documents/works/page/{last}").json()["orderedItems"]
if new > len(items) and last > 1:
    items = bc.get(f"/api/change_documents/works/page/{last - 1}").json()["orderedItems"] + items
made = [i["object"]["id"] for i in items[-new:]] if new > 0 else []
print(new, "work change(s)")
made

## Clean up
Deletes the works this batch made (and their instances). Skip this cell to keep them.

In [ ]:
for uri in made:
    uuid_ = uri.rsplit("/", 1)[1]
    print(uuid_, bc.delete(f"/api/works/{uuid_}", headers=AUTH).status_code)

## Loading from a URI
`/batches/` takes a URI the Airflow workers can read: a URL, or a path under `/opt/airflow/uploads`. It needs a `uri`; without one it answers `422`.

In [ ]:
show(bc.post("/api/batches/", json={}, headers=AUTH))
# bc.post("/api/batches/", json={"uri": "https://example.org/records.jsonld"}, headers=AUTH)